In [1]:
from itertools import cycle
import numpy as np
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import xgboost as xgb
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RepeatedKFold
from sklearn.preprocessing import StandardScaler 
from sklearn.datasets import make_regression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_curve, auc
from sklearn.metrics import make_scorer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from string import ascii_uppercase
from geopy.distance import geodesic
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn import linear_model
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import numpy as np
from xgboost import XGBRegressor
#from tqdm import tqdm
import time
from scipy.stats import uniform
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

In [2]:
import sys
import os

# Get the absolute path to your src folder
module_path = os.path.abspath(os.path.join('..', 'src'))

if module_path not in sys.path:
    sys.path.append(module_path)

from cleaning_and_helpers import plot_test_preds


In [3]:
np.random.seed(1298)

In [4]:
SI_X = np.load('../../pollenGeolocation_saved/SI_X_tax.npy')
SI_y = np.load('../../pollenGeolocation_saved/SI_y.npy')

FFAR_X = np.load('../../pollenGeolocation_saved/FFAR_X_tax.npy')
FFAR_y = np.load('../../pollenGeolocation_saved/FFAR_y.npy')

NCASI_X = np.load('../../pollenGeolocation_saved/NCASI_X_tax.npy')
NCASI_y = np.load('../../pollenGeolocation_saved/NCASI_y.npy')

In [5]:
# Train test split for each project
seed = 1298
test_size = 0.20

# --- Function to split each project ---
def split_project(X, y, test_size, random_state):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

# --- Split each project ---
SI_X_train, SI_X_test, SI_y_train, SI_y_test = split_project(SI_X, SI_y, test_size, seed)
NCASI_X_train, NCASI_X_test, NCASI_y_train, NCASI_y_test = split_project(NCASI_X, NCASI_y, test_size, seed)
FFAR_X_train, FFAR_X_test, FFAR_y_train, FFAR_y_test = split_project(FFAR_X, FFAR_y, test_size, seed)

# --- Concatenate train and test splits from all projects ---
X_train = np.concatenate([SI_X_train, NCASI_X_train, FFAR_X_train], axis=0)
y_train = np.concatenate([SI_y_train, NCASI_y_train, FFAR_y_train], axis=0)

X_test = np.concatenate([SI_X_test, NCASI_X_test, FFAR_X_test], axis=0)
y_test = np.concatenate([SI_y_test, NCASI_y_test, FFAR_y_test], axis=0)

# --- Standardize X and y using training data only ---
sc_X = StandardScaler()
sc_y = StandardScaler()

X_train = sc_X.fit_transform(X_train)
y_train = sc_y.fit_transform(y_train)

X_test = sc_X.transform(X_test)
y_test = sc_y.transform(y_test)


In [ ]:
# -----------------------------
# Custom RMSE scorer
# -----------------------------
def multioutput_rmse(y_true, y_pred):
    return np.sqrt(((y_true - y_pred) ** 2).mean())

rmse_scorer = make_scorer(multioutput_rmse, greater_is_better=False)

# -----------------------------
# Cross-validation setup
# -----------------------------
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=1298)

# -----------------------------
# Models and param grids
# -----------------------------
models = {
    "MultiTaskLasso": (
        linear_model.MultiTaskLasso(max_iter=10000),
        {
            'estimator__alpha': np.arange(0.00, 1.0, 0.01)
        }
    ),

    "SVR": (
        MultiOutputRegressor(SVR()),
        {
            'estimator__estimator__C': uniform(0.1, 10),  # Uniform distribution between 0.1 and 10
            'estimator__estimator__kernel': ['linear', 'rbf', 'poly'],
            'estimator__estimator__gamma': ['scale', 'auto']
        }  
    ),

    "KNN": (
        MultiOutputRegressor(KNeighborsRegressor()),
        {
            'estimator__estimator__n_neighbors': np.arange(2, 50, 1),  # Number of neighbors (2 to 50)
            'estimator__estimator__weights': ['uniform', 'distance'],  # Weighting method
            'estimator__estimator__p': [1, 2],  # Manhattan or Euclidean distance
            'estimator__estimator__algorithm': ['auto']  # Default to 'auto'
        }
    ),

    "DecisionTree": (
        MultiOutputRegressor(DecisionTreeRegressor(random_state=seed)),
        {
            'estimator__estimator__max_depth': [None] + list(range(3, 21, 1)),      # Depths from 3 to 20 + unlimited depth
            'estimator__estimator__min_samples_split': [2, 5, 10, 20, 50],          # Minimum samples to split
            'estimator__estimator__min_samples_leaf': [1, 2, 5, 10, 20, 50],        # Minimum samples in a leaf
            'estimator__estimator__max_features': ['auto', 'sqrt', 'log2', None],   # Features considered at each split
            'estimator__estimator__max_leaf_nodes': [None] + list(range(10, 101, 10))  # Maximum number of leaf nodes

        }
    ),

    "RandomForest": (
        MultiOutputRegressor(RandomForestRegressor(random_state=seed)),
        {
            'estimator__estimator__n_estimators': [50, 100, 200, 500, 1000],           # Number of trees
            'estimator__estimator__max_depth': [None] + list(range(10, 51, 10)),   # Maximum depth of trees
            'estimator__estimator__min_samples_split': [2, 5, 10, 20, 50],             # Minimum samples to split a node
            'estimator__estimator__min_samples_leaf': [1, 2, 5, 10, 20, 50],          # Minimum samples in a leaf node
            'estimator__estimator__max_features': ['auto', 'sqrt', 'log2', None],  # Features considered for splits
            #'bootstrap': [True],                      # Whether to use bootstrapping
            #'estimator__estimator__max_samples': [0.5, 0.75, None]                 # Optional: Samples per tree if bootstrap=True

        }  
    ),

    "XGBoost": (
        MultiOutputRegressor(XGBRegressor(objective='reg:squarederror', random_state=seed)),
        {
            'estimator__estimator__n_estimators': [100, 200, 500, 1000],          # Number of trees
            'estimator__estimator__learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],  # Learning rate
            'estimator__estimator__max_depth': [3, 5, 7, 10, 15, 20],            # Maximum tree depth
            'estimator__estimator__min_child_weight': [1, 5, 10, 20],            # Minimum child weight
            'estimator__estimator__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],        # Subsampling ratio
            'estimator__estimator__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], # Features per tree
            'estimator__estimator__gamma': [0, 0.1, 0.2, 0.5, 1, 5],             # Minimum loss reduction
            'estimator__estimator__reg_alpha': [0, 0.01, 0.1, 1, 10, 100],       # L1 regularization
            'estimator__estimator__reg_lambda': [1, 10, 50, 100]                 # L2 regularization

        }  
    )
}



# -----------------------------
# Run GridSearchCV for all models
# -----------------------------
results = {}

for name, (model, param_grid) in models.items():
    print(f"Training model: {name}")
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('estimator', model)
    ])
    
    # search = GridSearchCV(
    #     pipeline,
    #     param_grid=param_grid,
    #     scoring=rmse_scorer,
    #     cv=cv,
    #     n_jobs=-1,
    #     verbose=1
    # )

    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,   # NOTE: use param_distributions, not param_grid
        scoring=rmse_scorer,
        cv=cv,
        n_iter=50,         # <-- you choose how many random combos to try
        n_jobs=-1,
        verbose=1,
        random_state=1298
    )
    
    search.fit(X_train, y_train)
    results[name] = search

#--------------------------
# Compare Results
#--------------------------
for name, search in results.items():
    print(f"{name}: Best CV RMSE = {-search.best_score_:.3f}")

Training model: MultiTaskLasso
Fitting 15 folds for each of 50 candidates, totalling 750 fits


In [ ]:
for name, search in results.items():
    print(f"{name} best params:")
    print(search.best_params_)
    print()

MultiTaskLasso best params:
{'estimator__alpha': 0.08}

SVR best params:
{'estimator__estimator__C': 2.847162213648956, 'estimator__estimator__gamma': 'auto', 'estimator__estimator__kernel': 'rbf'}

KNN best params:
{'estimator__estimator__weights': 'uniform', 'estimator__estimator__p': 2, 'estimator__estimator__n_neighbors': 3, 'estimator__estimator__algorithm': 'auto'}

DecisionTree best params:
{'estimator__estimator__min_samples_split': 10, 'estimator__estimator__min_samples_leaf': 2, 'estimator__estimator__max_leaf_nodes': 40, 'estimator__estimator__max_features': None, 'estimator__estimator__max_depth': 18}

RandomForest best params:
{'estimator__estimator__n_estimators': 200, 'estimator__estimator__min_samples_split': 5, 'estimator__estimator__min_samples_leaf': 1, 'estimator__estimator__max_features': 'log2', 'estimator__estimator__max_depth': None}

XGBoost best params:
{'estimator__estimator__subsample': 0.8, 'estimator__estimator__reg_lambda': 1, 'estimator__estimator__reg_a

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, median_absolute_error
from geopy.distance import geodesic
import numpy as np

def evaluate_model(name, model_class, best_params, X_train, y_train, X_test, y_test):
    """
    Initialize, fit, predict, and evaluate a model using best hyperparameters.

    Parameters:
    - name: string, name of model
    - model_class: class or callable that returns a model
    - best_params: dict of hyperparameters
    - X_train, y_train, X_test, y_test: data splits

    Returns:
    - dict with performance metrics and predictions
    """
    print(f"Evaluating {name}...")

    # If model_class is a callable (e.g., lambda), call it to get the estimator
    model = model_class(**best_params)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    r2 = r2_score(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    mae = median_absolute_error(y_test, preds)
    distances = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds)]
    avg_dist = np.mean(distances)
    se_dist = np.std(distances)

    return {
        "model": name,
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
        "avg_km_error": avg_dist,
        "se_km_error": se_dist,
        "preds": preds
    }


In [ ]:
from sklearn.linear_model import MultiTaskLasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

# Define model mapping (must match what was passed into GridSearchCV)
model_classes = {
    "MultiTaskLasso": MultiTaskLasso,
    "SVR": lambda **kwargs: MultiOutputRegressor(SVR(**kwargs)),
    "KNN": lambda **kwargs: MultiOutputRegressor(KNeighborsRegressor(**kwargs)),
    "DecisionTree": lambda **kwargs: MultiOutputRegressor(DecisionTreeRegressor( **kwargs)),
    "RandomForest": lambda **kwargs: MultiOutputRegressor(RandomForestRegressor( **kwargs)),
    "XGBoost": lambda **kwargs: MultiOutputRegressor(XGBRegressor(objective="reg:squarederror", **kwargs))
}

# Collect results
all_metrics = []

for name, search in results.items():
    best_params = search.best_params_
    
    # Flatten parameter dict for instantiation (remove 'estimator__estimator__' prefix)
    flat_params = {
        k.replace("estimator__estimator__", "").replace("estimator__", ""): v
        for k, v in best_params.items()
    }

    metrics = evaluate_model(
        name=name,
        model_class=model_classes[name],
        best_params=flat_params,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )
    all_metrics.append(metrics)


Evaluating MultiTaskLasso...
Evaluating SVR...
Evaluating KNN...
Evaluating DecisionTree...
Evaluating RandomForest...
Evaluating XGBoost...


In [ ]:
import pandas as pd

results_df = pd.DataFrame([{
    "Model": m["model"],
    "R²": m["r2"],
    "RMSE": m["rmse"],
    "MAE": m["mae"],
    "Avg Geodesic Error (km)": m["avg_km_error"],
    "SE Geodesic Error (km)": m["se_km_error"]
} for m in all_metrics])

results_df.to_csv("../tables/taxonomic/tax_best_case_tuned_results.csv", index=False)

results_df


,Model,R²,RMSE,MAE,Avg Geodesic Error (km),SE Geodesic Error (km)
0,MultiTaskLasso,0.849040,0.397274,0.074763,36.147007,51.139706
1,SVR,0.881733,0.350300,0.071784,27.697731,48.089278
2,KNN,0.976115,0.152536,0.015578,10.163496,22.885707
3,DecisionTree,0.866810,0.373581,0.023184,19.207829,55.389118
4,RandomForest,0.952025,0.219919,0.019599,15.439813,31.887088
5,XGBoost,0.915419,0.298037,0.021084,17.296902,43.440391
